In [1]:
import numpy as np
import pandas as pd
import os
import joblib
from sklearn.model_selection import train_test_split
import json

In [2]:
# ================= 配置参数 =================
PROCESSED_DATA_DIR = "./processed_data_rpm_only/"
OUTPUT_DIR = "./benchmark_dataset_rpm_offset_100/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 数据集划分比例
TRAIN_RATIO = 0.6
VAL_RATIO = 0.2
TEST_RATIO = 0.2

# 随机种子（保证可复现）
RANDOM_SEED = 42

# 真实故障检测阈值（基于Z-score，可根据数据调整）
REAL_ANOMALY_THRESHOLDS = {
    'ENGINE_RPM': 3.5,           # RPM的Z-score阈值
    'COOLANT_TEMPERATURE': 3.0,  # 温度的Z-score阈值
    'CONTROL_MODULE_VOLTAGE': 3.0, # 电压的Z-score阈值
}

# 传感器冻结检测参数
FREEZE_THRESHOLD = 0.001  # 连续相同值的容忍度

In [3]:
# 加载预处理数据
print("加载预处理数据...")
windowed_data = np.load(os.path.join(PROCESSED_DATA_DIR, "windowed_data.npy"))
windowed_scenarios = np.load(os.path.join(PROCESSED_DATA_DIR, "windowed_scenarios.npy"), allow_pickle=True)
scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, "scaler.pkl"))

# 加载特征列名
with open(os.path.join(PROCESSED_DATA_DIR, "feature_columns.json"), 'r') as f:
    FEATURE_COLS = json.load(f)

print(f"数据形状: {windowed_data.shape}")
print(f"特征数量: {len(FEATURE_COLS)}")

加载预处理数据...
数据形状: (30424, 60, 1)
特征数量: 1


In [4]:
# ================= 真实故障检测函数 =================

def detect_real_rpm_spike(data, threshold=3.5):
    """
    检测真实的RPM尖峰异常
    使用Z-score方法检测RPM的异常高值
    """
    rpm_idx = FEATURE_COLS.index('ENGINE_RPM ()')
    n_samples = data.shape[0]
    anomaly_flags = np.zeros(n_samples, dtype=bool)
    
    for i in range(n_samples):
        window_rpm = data[i, :, rpm_idx]  
        # 计算窗口内RPM的Z-score
        mean_rpm = np.mean(window_rpm)
        std_rpm = np.std(window_rpm)
        
        if std_rpm > 0:
            # 找出窗口内超过阈值的点
            z_scores = np.abs((window_rpm - mean_rpm) / std_rpm)
            # 如果窗口内任一时刻RPM超过阈值，标记为异常
            if np.any(z_scores > threshold):
                anomaly_flags[i] = True
    
    return anomaly_flags

def detect_real_coolant_drift(data, threshold=3.0):
    """
    检测真实冷却液温度漂移异常
    检测温度持续上升或异常高值
    """
    temp_idx = FEATURE_COLS.index('COOLANT_TEMPERATURE ()')
    n_samples = data.shape[0]
    anomaly_flags = np.zeros(n_samples, dtype=bool)
    
    for i in range(n_samples):
        window_temp = data[i, :, temp_idx]
        
        # 检测1: 温度是否在窗口内显著上升（线性趋势）
        x = np.arange(len(window_temp))
        if len(window_temp) > 1:
            # 计算线性回归斜率
            slope = np.polyfit(x, window_temp, 1)[0]
            # 如果斜率大于正阈值，说明温度持续上升
            if slope > 0.05:  # 可调整阈值
                anomaly_flags[i] = True
                continue
        
        # 检测2: 温度是否超过3倍标准差
        mean_temp = np.mean(window_temp)
        std_temp = np.std(window_temp)
        if std_temp > 0:
            # 检测是否有温度值超过阈值
            z_scores = np.abs((window_temp - mean_temp) / std_temp)
            if np.any(z_scores > threshold):
                anomaly_flags[i] = True
    
    return anomaly_flags

def detect_real_battery_drop(data, threshold=3.0):
    """
    检测真实电池电压下降异常
    检测电压持续下降或异常低值
    """
    volt_idx = FEATURE_COLS.index('CONTROL_MODULE_VOLTAGE ()')
    n_samples = data.shape[0]
    anomaly_flags = np.zeros(n_samples, dtype=bool)
    
    for i in range(n_samples):
        window_volt = data[i, :, volt_idx]
        
        # 检测1: 电压是否在窗口内显著下降（线性趋势）
        x = np.arange(len(window_volt))
        if len(window_volt) > 1:
            slope = np.polyfit(x, window_volt, 1)[0]
            # 如果斜率小于负阈值，说明电压持续下降
            if slope < -0.05:  # 可调整阈值
                anomaly_flags[i] = True
                continue
        
        # 检测2: 电压是否低于阈值
        mean_volt = np.mean(window_volt)
        std_volt = np.std(window_volt)
        if std_volt > 0:
            z_scores = np.abs((window_volt - mean_volt) / std_volt)
            if np.any(z_scores > threshold):
                anomaly_flags[i] = True
    
    return anomaly_flags

def detect_real_sensor_freeze(data, freeze_threshold=0.001):
    """
    检测真实传感器冻结异常
    检测特征值是否长时间保持不变
    """
    n_samples = data.shape[0]
    anomaly_flags = np.zeros(n_samples, dtype=bool)
    
    # 检测所有特征的冻结
    for feat_name in ['ENGINE_RPM ()', 'VEHICLE_SPEED ()', 'COOLANT_TEMPERATURE ()']:
        if feat_name not in FEATURE_COLS:
            continue
            
        feat_idx = FEATURE_COLS.index(feat_name)
        
        for i in range(n_samples):
            window_feat = data[i, :, feat_idx]
            
            # 计算连续相同值的比例
            # 检查连续值的变化
            diff = np.abs(np.diff(window_feat))
            # 如果大部分差异都很小（< freeze_threshold），可能是冻结
            frozen_ratio = np.sum(diff < freeze_threshold) / len(diff)
            
            # 如果超过80%的值几乎相同，判定为冻结
            if frozen_ratio > 0.8:
                anomaly_flags[i] = True
                break  # 一个特征冻结就标记为异常
    
    return anomaly_flags

def detect_real_anomalies(data):
    """  
    综合检测所有真实异常
    返回: (anomaly_flags, anomaly_types)
    """
    n_samples = data.shape[0]
    anomaly_flags = np.zeros(n_samples, dtype=bool)
    anomaly_types = np.array(['normal'] * n_samples, dtype=object)
    
    print("  检测RPM尖峰...")
    rpm_anomalies = detect_real_rpm_spike(data)
    anomaly_flags = np.logical_or(anomaly_flags, rpm_anomalies)
    anomaly_types[rpm_anomalies] = 'real_rpm_spike'
    
    print("  检测冷却液温度漂移...")
    coolant_anomalies = detect_real_coolant_drift(data)
    anomaly_flags = np.logical_or(anomaly_flags, coolant_anomalies)
    # 如果已标记为其他异常，保留第一个异常类型
    mask = coolant_anomalies & (anomaly_types == 'normal')    
    anomaly_types[mask] = 'real_coolant_drift'
    
    print("  检测电池电压下降...")
    battery_anomalies = detect_real_battery_drop(data)
    anomaly_flags = np.logical_or(anomaly_flags, battery_anomalies)
    mask = battery_anomalies & (anomaly_types == 'normal')
    anomaly_types[mask] = 'real_battery_drop'
    
    print("  检测传感器冻结...")
    freeze_anomalies = detect_real_sensor_freeze(data)
    anomaly_flags = np.logical_or(anomaly_flags, freeze_anomalies)
    mask = freeze_anomalies & (anomaly_types == 'normal')
    anomaly_types[mask] = 'real_sensor_freeze'
    
    # 对于同时有多种异常的样本，标记为'complex_real_anomaly'
    multi_anomaly_mask = (anomaly_flags) & (anomaly_types == 'normal')
    # 但这里不会触发，因为上面的逻辑已经标记了类型
    
    return anomaly_flags, anomaly_types

In [5]:
# ================= 故障注入函数 =================

def inject_rpm_spike(data, start, end, scale_factor=3.0):
    """RPM突增: 将指定窗口的RPM值乘以系数"""
    rpm_idx = FEATURE_COLS.index('ENGINE_RPM ()')
    # data[start:end, :, rpm_idx] = data[start:end, :, rpm_idx] * scale_factor
    data[start:end, :, rpm_idx] += 100
    return data

def inject_coolant_drift(data, start, end, drift_rate=0.05):
    """冷却液温度漂移: 在窗口内线性增加温度"""
    temp_idx = FEATURE_COLS.index('COOLANT_TEMPERATURE ()')
    for i in range(start, end):
        drift_amount = (i - start) * drift_rate
        data[i, :, temp_idx] = data[i, :, temp_idx] + drift_amount
    return data

def inject_battery_drop(data, start, end, drop_rate=-0.02):
    """电池电压下降: 在窗口内线性降低电压"""
    volt_idx = FEATURE_COLS.index('CONTROL_MODULE_VOLTAGE ()')
    for i in range(start, end):
        drop_amount = (i - start) * drop_rate
        data[i, :, volt_idx] = data[i, :, volt_idx] + drop_amount
    return data

def inject_sensor_freeze(data, start, end, freeze_feature='ENGINE_RPM ()'):
    """传感器冻结: 将指定特征在窗口内的值固定为窗口第一个值"""
    feat_idx = FEATURE_COLS.index(freeze_feature)
    for i in range(start, end):
        first_val = data[i, 0, feat_idx].copy()
        data[i, :, feat_idx] = first_val
    return data

def inject_synthetic_anomalies(data, fault_config):
    """
    向数据注入合成故障
    返回: (data_with_anomalies, labels, fault_types)  
    """
    data_with_anomalies = data.copy()
    labels = np.zeros(data.shape[0], dtype=int)
    fault_types = np.array(['normal'] * data.shape[0], dtype=object)
    
    for fault_type, start, end in fault_config:
        # 检查索引是否在范围内
        if start >= data.shape[0] or end > data.shape[0]:
            print(f"  警告: 故障 {fault_type} 的索引范围 {start}-{end} 超出数据范围 (0-{data.shape[0]})，跳过。")
            continue
        
        print(f"  注入 {fault_type} 在样本 {start} 到 {end}")
        
        if fault_type == "rpm_spike":
            data_with_anomalies = inject_rpm_spike(data_with_anomalies, start, end)
        elif fault_type == "coolant_drift":
            data_with_anomalies = inject_coolant_drift(data_with_anomalies, start, end)
        elif fault_type == "battery_drop":
            data_with_anomalies = inject_battery_drop(data_with_anomalies, start, end)
        elif fault_type == "sensor_freeze":
            data_with_anomalies = inject_sensor_freeze(data_with_anomalies, start, end, freeze_feature='ENGINE_RPM ()')
        else:
            print(f"  未知故障类型: {fault_type}")
            continue
        
        # 更新标签
        labels[start:end] = 1
        fault_types[start:end] = fault_type
    
    return data_with_anomalies, labels, fault_types

In [6]:
# ================= 保存数据集 =================

def convert_numpy_to_python(obj):
    """
    递归地将 NumPy 类型转换为 Python 原生类型，以便 JSON 序列化
    """
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {key: convert_numpy_to_python(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_to_python(item) for item in obj]
    elif isinstance(obj, tuple):
        return tuple(convert_numpy_to_python(item) for item in obj)
    elif isinstance(obj, (str, int, float, bool)):
        return obj
    else:
        # 如果是其他类型，尝试转换为字符串
        return str(obj)

In [7]:
# ================= 主执行流程 =================
if __name__ == "__main__":
    print("\n=== 步骤1: 划分数据集 ===")
    total_samples = windowed_data.shape[0]
    indices = np.arange(total_samples)
    
    # 先划分训练集和临时集（验证+测试）
    train_indices, temp_indices = train_test_split(
        indices, 
        train_size=TRAIN_RATIO, 
        random_state=RANDOM_SEED,
        shuffle=True
    )
    
    # 再从临时集中划分验证集和测试集
    val_size = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
    val_indices, test_indices = train_test_split(
        temp_indices,
        train_size=val_size,
        random_state=RANDOM_SEED,
        shuffle=True
    )
    
    print(f"总样本数: {total_samples}")
    print(f"训练集: {len(train_indices)} 样本 ({len(train_indices)/total_samples*100:.1f}%)")
    print(f"验证集: {len(val_indices)} 样本 ({len(val_indices)/total_samples*100:.1f}%)")
    print(f"测试集: {len(test_indices)} 样本 ({len(test_indices)/total_samples*100:.1f}%)")
    
    # 提取数据集
    train_data = windowed_data[train_indices]
    train_scenarios = windowed_scenarios[train_indices]
    
    val_data = windowed_data[val_indices]
    val_scenarios = windowed_scenarios[val_indices]
    
    test_data = windowed_data[test_indices]
    test_scenarios = windowed_scenarios[test_indices]
    
    print("\n=== 步骤2: 注入合成故障 (仅在验证集和测试集) ===")
    
    val_fault_config = [
        ("rpm_spike", 1000, 1030),
        # ("coolant_drift", 2000, 2120),
        # ("battery_drop", 3000, 3090),
        # ("sensor_freeze", 4000, 4040),
    ]
    
    test_fault_config = [
        ("rpm_spike", 500, 530),
        # ("coolant_drift", 1500, 1620),
        # ("battery_drop", 2500, 2590),
        # ("sensor_freeze", 3500, 3540),
    ]
    
    # 验证集注入
    print("\n验证集合成故障注入:")
    val_data_final, val_labels, val_fault_types = inject_synthetic_anomalies(
        val_data, val_fault_config
    )
    
    # 测试集注入
    print("\n测试集合成故障注入:")
    test_data_final, test_labels, test_fault_types = inject_synthetic_anomalies(
        test_data, test_fault_config
    )
    
    # 训练集保持原始数据，并且全部标记为 normal
    train_data_final = train_data.copy()
    train_labels = np.zeros(len(train_data), dtype=int)
    train_fault_types = np.array(
        ['normal'] * len(train_data),
        dtype=object
    )
    
    print("\n=== 步骤3: 最终数据统计 ===")
    
    print(
        f"训练集: 正常={np.sum(train_labels == 0)}, "
        f"异常={np.sum(train_labels == 1)}"
    )
    
    print(
        f"验证集: 正常={np.sum(val_labels == 0)}, "
        f"异常={np.sum(val_labels == 1)}"
    )
    if np.sum(val_labels == 1) > 0:
        print("  验证集异常类型:")
        type_counts = pd.Series(
            val_fault_types[val_labels == 1]
        ).value_counts()
        for t, count in type_counts.items():
            print(f"    {t}: {count}")
    
    print(
        f"测试集: 正常={np.sum(test_labels == 0)}, "
        f"异常={np.sum(test_labels == 1)}"
    )
    if np.sum(test_labels == 1) > 0:
        print("  测试集异常类型:")
        type_counts = pd.Series(
            test_fault_types[test_labels == 1]
        ).value_counts()
        for t, count in type_counts.items():
            print(f"    {t}: {count}")


=== 步骤1: 划分数据集 ===
总样本数: 30424
训练集: 18254 样本 (60.0%)
验证集: 6085 样本 (20.0%)
测试集: 6085 样本 (20.0%)

=== 步骤2: 注入合成故障 (仅在验证集和测试集) ===

验证集合成故障注入:
  注入 rpm_spike 在样本 1000 到 1030

测试集合成故障注入:
  注入 rpm_spike 在样本 500 到 530

=== 步骤3: 最终数据统计 ===
训练集: 正常=18254, 异常=0
验证集: 正常=6055, 异常=30
  验证集异常类型:
    rpm_spike: 30
测试集: 正常=6055, 异常=30
  测试集异常类型:
    rpm_spike: 30


In [8]:
# 保存划分信息（使用转换函数）
print("\n=== 步骤6: 保存数据集 ===")

# 保存训练集
train_df = pd.DataFrame({
    'sample_id': np.arange(len(train_data_final)),
    'label': train_labels,
    'fault_type': train_fault_types,
    'scenario': train_scenarios
})
train_df.to_parquet(os.path.join(OUTPUT_DIR, "train.parquet"), index=False)  
np.save(os.path.join(OUTPUT_DIR, "train_data.npy"), train_data_final)

# 保存验证集
val_df = pd.DataFrame({
    'sample_id': np.arange(len(val_data_final)),
    'label': val_labels,
    'fault_type': val_fault_types,
    'scenario': val_scenarios
})
val_df.to_parquet(os.path.join(OUTPUT_DIR, "validation_benchmark.parquet"), index=False)
np.save(os.path.join(OUTPUT_DIR, "validation_data.npy"), val_data_final)

# 保存测试集
test_df = pd.DataFrame({
    'sample_id': np.arange(len(test_data_final)),
    'label': test_labels,
    'fault_type': test_fault_types,
    'scenario': test_scenarios
})
test_df.to_parquet(os.path.join(OUTPUT_DIR, "test_benchmark.parquet"), index=False)
np.save(os.path.join(OUTPUT_DIR, "test_data.npy"), test_data_final)

# 构建统计信息（确保所有数值都是Python原生类型）
train_anomaly_counts = pd.Series(train_fault_types[train_labels==1]).value_counts()
val_anomaly_counts = pd.Series(val_fault_types[val_labels==1]).value_counts()
test_anomaly_counts = pd.Series(test_fault_types[test_labels==1]).value_counts()

# 使用转换函数确保所有数据可JSON序列化
split_info = {
    'total_samples': int(total_samples),
    'train_samples': int(len(train_data)),
    'validation_samples': int(len(val_data)),
    'test_samples': int(len(test_data)),
    'train_ratio': TRAIN_RATIO,
    'val_ratio': VAL_RATIO,
    'test_ratio': TEST_RATIO,
    'random_seed': RANDOM_SEED,
    'feature_columns': FEATURE_COLS,
    'val_fault_config': val_fault_config,
    'test_fault_config': test_fault_config,
    'real_anomaly_thresholds': REAL_ANOMALY_THRESHOLDS,
    'statistics': {
        'train': {
            'total': int(len(train_data)),
            'normal': int(np.sum(train_labels==0)),
            'anomaly': int(np.sum(train_labels==1)),
            'anomaly_types': convert_numpy_to_python(dict(train_anomaly_counts))
        },
        'validation': {
            'total': int(len(val_data)),
            'normal': int(np.sum(val_labels==0)),
            'anomaly': int(np.sum(val_labels==1)),
            'anomaly_types': convert_numpy_to_python(dict(val_anomaly_counts))
        },
        'test': {
            'total': int(len(test_data)),
            'normal': int(np.sum(test_labels==0)),
            'anomaly': int(np.sum(test_labels==1)),
            'anomaly_types': convert_numpy_to_python(dict(test_anomaly_counts))
        }
    }
}

# 保存为JSON
with open(os.path.join(OUTPUT_DIR, "dataset_info.json"), 'w') as f:
    json.dump(split_info, f, indent=2)

print(f"\n✅ 基准数据集已生成，保存在 {OUTPUT_DIR}")
print("\n生成的文件:")
print("  - train.parquet (训练集标签和元数据)")
print("  - train_data.npy (训练集数据)")
print("  - validation_benchmark.parquet (验证集标签和元数据)")
print("  - validation_data.npy (验证集数据)")
print("  - test_benchmark.parquet (测试集标签和元数据)")
print("  - test_data.npy (测试集数据)")
print("  - dataset_info.json (数据集划分信息)")

print("\n=== 数据集使用说明 ===")
print("1. 训练集 (train_data.npy): 包含真实异常 + 正常数据，用于学习正常模式")
print("2. 验证集 (validation_data.npy): 包含真实异常 + 合成异常，用于调优阈值")
print("3. 测试集 (test_data.npy): 包含真实异常 + 合成异常，用于最终评估")
print("\n加载数据示例:")
print("  train_data = np.load('train_data.npy')")
print("  train_df = pd.read_parquet('train.parquet')")
print(f"\n真实故障检测阈值: {REAL_ANOMALY_THRESHOLDS}")


=== 步骤6: 保存数据集 ===

✅ 基准数据集已生成，保存在 ./benchmark_dataset_rpm_offset_100/

生成的文件:
  - train.parquet (训练集标签和元数据)
  - train_data.npy (训练集数据)
  - validation_benchmark.parquet (验证集标签和元数据)
  - validation_data.npy (验证集数据)
  - test_benchmark.parquet (测试集标签和元数据)
  - test_data.npy (测试集数据)
  - dataset_info.json (数据集划分信息)

=== 数据集使用说明 ===
1. 训练集 (train_data.npy): 包含真实异常 + 正常数据，用于学习正常模式
2. 验证集 (validation_data.npy): 包含真实异常 + 合成异常，用于调优阈值
3. 测试集 (test_data.npy): 包含真实异常 + 合成异常，用于最终评估

加载数据示例:
  train_data = np.load('train_data.npy')
  train_df = pd.read_parquet('train.parquet')

真实故障检测阈值: {'ENGINE_RPM': 3.5, 'COOLANT_TEMPERATURE': 3.0, 'CONTROL_MODULE_VOLTAGE': 3.0}
